# Problem 5.

**Total: 90 points.**

## The Tide Ledger

This 14-part arc follows one fresh harbor-town corpus from tokens to a compressed similarity model. Run the notebook in order: every later part uses names established earlier.


In [ ]:
import os
import pathlib

_root = next(
    path for path in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import gensim.downloader
import numpy as np
from gensim.utils import simple_preprocess

corpus_path = _root / "mocktests" / "r1-001" / "data" / "corpus.txt"
corpus = corpus_path.read_text(encoding="utf-8")
kv = gensim.downloader.load("glove-wiki-gigaword-100")
print(f"Loaded {len(corpus.split())} whitespace-separated words from {corpus_path.name}.")


## Part 5.1 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Tokenize the supplied corpus with gensim.utils.simple_preprocess. Store the list in tokens, the number of token occurrences in token_count, and the number of distinct token types in type_count. Print both counts.


In [ ]:
tokens = ...
token_count = ...
type_count = ...
print("token count:", token_count)
print("type count:", type_count)


## Part 5.2 (5 points)

**Type:** theory · **Answer form: short-answer**  
**Flag:** Coding not required.

A classmate runs the supplied cell below in two fresh Python processes and sees the same counts but a different preview order. Explain why the two census prints can differ after list(set(tokens)). State one other piece of information that conversion to a set loses.


In [ ]:
deduplicated_with_set = list(set(tokens))
print("set census:", len(deduplicated_with_set), deduplicated_with_set[:8])


**Your short answer:** ...


## Part 5.3 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Consume tokens from Part 5.1. Preserve first-occurrence order while removing duplicates, filter the result to words in kv.key_to_index, and store it as embedded_tokens. Look up every retained vector and store the vectors in embedded_vectors. At each lookup, apply the course boundary cast np.asarray(..., dtype=np.float64). Construct oov_tokens even though it may be empty; the filter code itself is graded by inspection.


In [ ]:
ordered_types = ...
embedded_tokens = ...
oov_tokens = ...
embedded_vectors = ...
print("embedded types:", len(embedded_tokens), "| OOV types:", len(oov_tokens))


## Part 5.4 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Stack embedded_vectors from Part 5.3 into the float64 matrix W_raw, with one retained token per row and exactly 100 columns. The identifier and orientation are required. Assert its shape and dtype.


In [ ]:
W_raw = ...
N = len(embedded_tokens)
assert W_raw.shape == (N, 100)
assert W_raw.dtype == np.float64


## Part 5.5 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.  
**Ban (zero points for this part):** Do not use loops or any np.linalg function.

Consume W_raw. Compute each row's squared length in row_sq, its length in row_norms with shape (N, 1), and the unit-row matrix W by broadcasting. Keep all three required identifiers.


In [ ]:
row_sq = ...
row_norms = ...
W = ...
assert row_sq.shape == (N,)
assert row_norms.shape == (N, 1)
assert W.shape == W_raw.shape
assert np.allclose((W * W).sum(axis=1), 1.0, atol=1e-12, rtol=0)


## Part 5.6 (5 points)

**Type:** theory · **Answer form: short-answer**  
**Flag:** Coding not required.

For any two rows of the unit-row matrix W from Part 5.5, give the exact range of their dot product. State precisely when each endpoint occurs.


**Your short answer:** ...


## Part 5.7 (15 points)

**Type:** theory · **Answer form: proof**  
**Flags:** Reasoning required; coding not allowed.

Define the fresh corpus similarity matrix by $S = WW^T$, using Part 5.5's rows $w_1$ through $w_N$. Express $S_{ij}$ entrywise as a sum over the 100 embedding coordinates. Then prove from that expression that $S$ is symmetric and every diagonal entry equals $1$. Cite the relevant property of $W$ at each step.


**Your proof:** ...


In [ ]:
# Supplied for later parts after you complete the proof.
S = W @ W.T
assert S.shape == (N, N)


## Part 5.8 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed; np.linalg.svd is explicitly allowed in this part.

Compute the thin SVD of Part 5.5's W with full_matrices=False, using U_thin, sigma, and Vt_thin. Report sigma and assert that it is non-increasing.


In [ ]:
U_thin, sigma, Vt_thin = ...
print("descending singular values:", sigma)
assert np.all(np.diff(sigma) <= 1e-12)


## Part 5.9 (5 points)

**Type:** theory · **Answer form: short-answer**  
**Flag:** Coding not required.

Let q = min(N, 100). For this exact (N, 100) matrix, state the shapes returned by thin SVD and by full SVD: U, sigma, and Vt in each case. Explain which output contains the q singular values of W, and which additional spectrum is needed for the N by N matrix S.


**Your short answer:** ...


## Part 5.10 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Consume W, S, and sigma. Compute the full SVD of W as U_full, sigma_full, Vt_full. Build lambda_full, the length-N eigenvalue vector for S, by squaring the singular values and padding all remaining entries with zeros. Then reconstruct S as S_spectral = U_full @ diag(lambda_full) @ U_full.T and verify the decomposition.


In [ ]:
U_full, sigma_full, Vt_full = ...
lambda_full = ...
S_spectral = ...
assert U_full.shape == (N, N)
assert lambda_full.shape == (N,)
assert np.allclose(S_spectral, S, atol=1e-10, rtol=0)


## Part 5.11 (15 points)

**Type:** theory · **Answer form: proof**  
**Flags:** Reasoning required; coding not allowed.

Using Part 5.10's spectral decomposition, let $S_r$ retain the first $r$ eigenpairs of $S$. Derive a formula for the relative squared Frobenius error $\lVert S-S_r\rVert_F^2 / \lVert S\rVert_F^2$ using only the singular values $\sigma$ of $W$. Your proof must justify why fourth powers appear and identify the numerator as a tail sum.


**Your proof:** ...


## Part 5.12 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.  
**Constraint:** Build no truncated matrix or factorization.

Consume Part 5.8's descending sigma and Part 5.11's identity. With exactly one np.cumsum call, compute rel_sq_errors, where entry r - 1 is the relative squared Frobenius error of S_r for every rank r = 1, ..., q.


In [ ]:
spectral_energy = ...
rel_sq_errors = ...
assert rel_sq_errors.shape == sigma.shape
assert np.all(np.diff(rel_sq_errors) <= 1e-12)


## Part 5.13 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Using Part 5.12's rel_sq_errors and error_budget = 0.08, find the smallest positive integer r_star whose error is at most the budget. Do not recompute an SVD. Keep the two supplied certificate asserts: they must prove feasibility and minimality.


In [ ]:
error_budget = 0.08
r_star = ...
assert rel_sq_errors[r_star - 1] <= error_budget
assert r_star == 1 or rel_sq_errors[r_star - 2] > error_budget
print("smallest feasible rank:", r_star)


## Part 5.14 (5 points)

**Type:** theory · **Answer form: short-answer**  
**Flag:** Coding not required.

Consume N and r_star. If the symmetric rank-r_star approximation is stored as U_full[:, :r_star] plus its r_star retained eigenvalues, how many floating-point numbers are stored, versus storing all of S? Give the general crossover inequality in N and r under which the factorization uses fewer numbers, then state whether Part 5.13's selected factorization is smaller.


**Your short answer:** ...
